In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch12-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "12-07-multiobjective-optimization.py"

# 多目標最適化

在 BACE 資料中同時比較實測抑制活性與結構中的重原子數。程式在獨立的 Python 程序中執行。試做練習時，把程式加在分析格末端，再重跑分析格與顯示格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os

import deepchem as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem

# BACE-1 抑制劑，實測 pIC50。Subramanian et al., DOI 10.1021/acs.jcim.6b00290
tasks, (data,), _ = dc.molnet.load_bace_regression(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
assert tasks == ["pIC50"]
records = []
for smiles, y, weight in zip(data.ids, data.y[:, 0], data.w[:, 0]):
    if weight > 0 and np.isfinite(y):
        mol = Chem.MolFromSmiles(smiles)
        records.append((Chem.MolToSmiles(mol), mol.GetNumHeavyAtoms(), float(y)))
entries = pd.DataFrame(records, columns=["smiles", "heavy", "pIC50"])
table = entries.groupby(["smiles", "heavy"], as_index=False)["pIC50"].mean()

# 查有沒有另一個結構在兩欄都不輸，且至少一欄嚴格勝出。
def dominated(row, pool):
    no_worse = (pool.heavy <= row.heavy) & (pool.pIC50 >= row.pIC50)
    better = (pool.heavy < row.heavy) | (pool.pIC50 > row.pIC50)
    return (no_worse & better).any()

front = table.loc[[not dominated(row, table) for row in table.itertuples()]]
front = front.sort_values(["heavy", "pIC50"])
print("observations / structures / frontier:", len(entries), len(table), len(front))
print(front[["heavy", "pIC50"]].to_string(index=False, float_format="{:.3f}".format))

fig, ax = plt.subplots(figsize=(7.2, 4.6), layout="constrained")
ax.scatter(table.heavy, table.pIC50, s=15, color="#a0b3bb", alpha=0.53, label="BACE structures")
ax.scatter(front.heavy, front.pIC50, s=47, marker="D", color="#ad411e", label="Non-dominated", zorder=3)
ax.set(xlabel="Heavy atoms per molecule (fewer preferred)", ylabel="Measured pIC50 (higher preferred)")
ax.legend(loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
out = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out / "12-07-bace-front.png", dpi=180)

In [ ]:
from IPython.display import Image, display

import os
environment = dict(os.environ, VECLIB_MAXIMUM_THREADS="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MPLBACKEND="Agg")
subprocess.run([python, str(script)], check=True, env=environment)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "12-07-bace-front.png")))